# 🔍 Tahap 1: Preprocessing & Validasi Dataset (Deteksi Sampah Daur Ulang)

**Proyek Replikasi Penelitian YOLOv8**  
Mata Kuliah: Kecerdasan Buatan  
Dataset: Recyclable Waste (Roboflow)

### Deskripsi Tahap Ini:
Pada notebook ini, kita akan melakukan langkah-langkah penyiapan data, antara lain:
1. Memeriksa keberadaan dataset di lokal.
2. Mengupdate path di file `data.yaml` secara dinamis agar program training berjalan lancar.
3. Menganalisis sebaran kelas sampah (kaca, kertas, logam, plastik) menggunakan visualisasi bar chart.
4. Menggambar bounding box koordinat YOLO pada beberapa sampel gambar untuk memverifikasi kebenaran anotasi.

### 🛠️ 1. Setup Environment dan Import Library
Jalankan cell di bawah ini untuk mengimpor pustaka yang dibutuhkan. Jika library belum terinstal, hilangkan tanda pagar pada perintah instalasi di sel pertama.

In [ ]:
# !pip install pyyaml opencv-python matplotlib numpy

import os
import yaml
import cv2
import glob
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

print("✅ Library berhasil diimpor!")

### 📝 2. Validasi Dataset dan Update data.yaml secara Otomatis
Kode berikut akan mendeteksi folder root proyek Anda dan meregenerasi file `data.yaml` menggunakan path absolut agar training bisa dipanggil dari direktori mana pun.

In [ ]:
# Tentukan path root proyek secara dinamis
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))
yaml_path = os.path.join(project_root, "data.yaml")

print(f"Folder root proyek: {project_root}")
print(f"Path data.yaml: {yaml_path}")

# Periksa apakah folder dataset train/valid/test ada
splits = ['train', 'valid', 'test']
missing_splits = []
for split in splits:
    path_to_check = os.path.join(project_root, split)
    if not os.path.exists(path_to_check):
        missing_splits.append(split)

if missing_splits:
    print(f"❌ Error: Folder berikut tidak ditemukan: {missing_splits}")
    print("Silakan ekstrak dataset ke root folder proyek Anda.")
else:
    print("✅ Semua folder dataset (train, valid, test) lengkap!")
    
    # Update data.yaml
    data_yaml = {
        'path': project_root.replace('\\', '/'),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': 4,
        'names': ['kaca', 'kertas', 'logam', 'plastik']
    }
    
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)
    print("✅ data.yaml berhasil dikonfigurasi secara absolut!")

### 📊 3. Analisis Distribusi Kelas Sampah
Mari kita hitung berapa banyak objek dari masing-masing kategori (`kaca`, `kertas`, `logam`, `plastik`) di setiap folder pembagian (*split*).

In [ ]:
class_names = ['kaca', 'kertas', 'logam', 'plastik']
split_counts = {}

for split in splits:
    labels_dir = os.path.join(project_root, split, 'labels')
    label_files = glob.glob(os.path.join(labels_dir, "*.txt"))
    
    classes_counter = Counter()
    for lbl_file in label_files:
        try:
            with open(lbl_file, 'r') as lf:
                for line in lf:
                    parts = line.strip().split()
                    if parts:
                        class_id = int(parts[0])
                        classes_counter[class_id] += 1
        except:
            continue
            
    split_counts[split] = [classes_counter.get(i, 0) for i in range(4)]
    print(f"\n📂 Split [{split.upper()}]:")
    for idx, c_name in enumerate(class_names):
        print(f"   - {c_name.capitalize()}: {split_counts[split][idx]} objek")

# Visualisasi Distribusi Kelas menggunakan Bar Chart
x = np.arange(len(class_names))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, split_counts['train'], width, label='Train', color='#4f46e5')
ax.bar(x, split_counts['valid'], width, label='Validation', color='#0ea5e9')
ax.bar(x + width, split_counts['test'], width, label='Test', color='#10b981')

ax.set_ylabel('Jumlah Objek', fontsize=12)
ax.set_title('Distribusi Kelas Sampah Daur Ulang dalam Dataset', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in class_names], fontsize=11)
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 📸 4. Visualisasi Sampah Beserta Bounding Box
Di bawah ini, kita akan membuat fungsi pembantu untuk memuat gambar dari dataset dan menggambar kotak koordinat anotasi YOLO asli di atas gambar.

In [ ]:
def plot_sample_annotations(split='train', num_samples=3):
    images_dir = os.path.join(project_root, split, 'images')
    labels_dir = os.path.join(project_root, split, 'labels')
    
    image_files = glob.glob(os.path.join(images_dir, "*.jpg")) + glob.glob(os.path.join(images_dir, "*.jpeg")) + glob.glob(os.path.join(images_dir, "*.png"))
    if not image_files:
        print("Tidak ada gambar ditemukan!")
        return
        
    # Pilih gambar secara random
    selected_images = np.random.choice(image_files, min(num_samples, len(image_files)), replace=False)
    
    colors = {
        0: (239, 68, 68),   # kaca -> Merah
        1: (59, 130, 246),  # kertas -> Biru
        2: (234, 179, 8),   # logam -> Kuning
        3: (34, 197, 94)    # plastik -> Hijau
    }
    
    plt.figure(figsize=(15, 6))
    
    for idx, img_path in enumerate(selected_images):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        
        # Baca label yang sesuai
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(labels_dir, f"{base_name}.txt")
        
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as lf:
                for line in lf:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        class_id = int(parts[0])
                        # YOLO format: x_center, y_center, width, height (normalized 0-1)
                        x_c, y_c, bw, bh = map(float, parts[1:5])
                        
                        # Ubah ke koordinat pixel absolut
                        x1 = int((x_c - bw/2) * w)
                        y1 = int((y_c - bh/2) * h)
                        x2 = int((x_c + bw/2) * w)
                        y2 = int((y_c + bh/2) * h)
                        
                        # Gambar Bounding Box dan Teks Label
                        color = colors.get(class_id, (128, 128, 128))
                        cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                        label_text = class_names[class_id].upper()
                        cv2.putText(img, label_text, (x1, max(y1 - 10, 20)), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
                                    
        plt.subplot(1, num_samples, idx + 1)
        plt.imshow(img)
        plt.title(f"Sampah: {base_name}", fontsize=11)
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

plot_sample_annotations(split='train', num_samples=3)

### 🏁 Kesimpulan Tahap 1
Dataset berhasil diperiksa dan diverifikasi secara visual. Anotasi YOLO terletak di posisi yang tepat pada gambar. 
File `data.yaml` juga telah siap dengan path absolut proyek, sehingga model siap dilatih pada tahap berikutnya!